# Clinic Case Study — Phase 2: Two-Stage Model

**Case study**: Community Health Clinic | **Phase**: 2 of 5

## Learning Objectives
By the end of this notebook you will be able to:
1. Model a two-stage sequential service process (registration → triage).
2. Explain why Little's Law applies to each stage independently and to the whole system.
3. Identify the bottleneck stage from utilisation statistics.
4. Show that the total mean time in system is **not** simply the sum of two single-stage M/M/1 waits.

---
> Phase 2 adds a registration desk before the triage nurse.
> The system is now a two-stage tandem queue — no longer analytically tractable in general.

In [ ]:
import sys
from pathlib import Path
# Ensure course/ is on sys.path so the case_studies package is importable
_root = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / 'simdes').is_dir())
_course = _root / 'course'
if str(_course) not in sys.path:
    sys.path.insert(0, str(_course))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from case_studies.clinic.clinic_model import run_clinic, ClinicParams
from simdes.analysis import confidence_interval

## System Description

Patients now flow through two stages:

```
Arrival → [Queue] → Registration → [Queue] → Triage Nurse → Exit
```

| Stage | Servers | Mean service time | Effective μ |
|---|---|---|---|
| Registration | 1 | 3 min | 1/3 patients/min |
| Triage | 1 | 8 min | 1/8 patients/min |

Arrival rate λ = 5/hr = 1/12 patients/min → ρ_reg = (1/12)/(1/3) = 0.25; ρ_triage = 0.667.

The **triage nurse** is the bottleneck (higher utilisation).

In [ ]:
p2 = ClinicParams(
    n_registration=1,
    n_nurses=1,
    n_exam_rooms=100,   # not the focus yet
    arrival_rate=5.0 / 60.0,
    reg_mean=3.0,
    triage_mean=8.0,
    exam_mean=0.1,
    sim_time=480.0,
)

df = run_clinic(p2, n_reps=30)
df.head()

In [ ]:
# Summary statistics
metrics = ['mean_wait_registration', 'mean_wait_triage', 'mean_total_time']
for col in metrics:
    m, lo, hi = confidence_interval(df[col].to_numpy())
    print(f'{col:30s}: {m:.2f} min  95% CI [{lo:.2f}, {hi:.2f}]')

In [ ]:
# Is the total wait approximately the sum of individual stage waits?
stage_sum = df['mean_wait_registration'].mean() + df['mean_wait_triage'].mean()
total     = df['mean_total_time'].mean()
service_sum = 3.0 + 8.0   # deterministic service times would give exactly this

print(f'Sum of mean stage waits:  {stage_sum:.2f} min')
print(f'Mean total time:          {total:.2f} min')
print(f'Sum of service times:     {service_sum:.2f} min')
print(f'Excess (queueing delay):  {total - service_sum:.2f} min')

In [ ]:
# Effect of adding a second registration clerk
p2b = ClinicParams(
    n_registration=2,   # <-- two registration clerks
    n_nurses=1,
    n_exam_rooms=100,
    arrival_rate=5.0 / 60.0,
    reg_mean=3.0,
    triage_mean=8.0,
    exam_mean=0.1,
    sim_time=480.0,
)
df2b = run_clinic(p2b, n_reps=30)

for col in metrics:
    m1, lo1, hi1 = confidence_interval(df[col].to_numpy())
    m2, lo2, hi2 = confidence_interval(df2b[col].to_numpy())
    print(f'{col:30s}  1 clerk: {m1:.2f} [{lo1:.2f}, {hi1:.2f}]  |  2 clerks: {m2:.2f} [{lo2:.2f}, {hi2:.2f}]')

## Discussion

Adding a second registration clerk barely changes the total time in system because
registration is **not the bottleneck** (ρ_reg = 0.25).  All the queueing delay comes
from the triage nurse.  This is the key insight of bottleneck analysis:
resources with low utilisation have little effect on overall performance.

**Little's Law check**: $L = \lambda W$
- λ ≈ 5/hr = 1/12 patients/min
- W = mean total time (from simulation)
- L should equal approximately λ × W

In [ ]:
lam = 5.0 / 60.0
W_hat = df['mean_total_time'].mean()
L_littles = lam * W_hat
L_counted = df['n_patients'].mean() / 480.0 * W_hat  # rough
print(f'W = {W_hat:.2f} min')
print(f"Little's Law L = λW = {L_littles:.3f} patients in system (on average)")

## Try It Yourself

1. Increase the arrival rate to λ = 7/hr.  Which stage is now more critical?
2. Use `run_clinic` with `sim_time=4800` (10 days).  Do the CIs narrow?
3. Define a scenario where adding a registration clerk would matter. What would the parameters be?